# **TASK 04 — GRAPH NEURAL NETWORK DEVELOPMENT**

## **Imports & Environment Setup**

In [1]:
import os
import time
import pickle
import random

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, SAGEConv

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch version: 2.3.1+cpu
CUDA available: False


## **Reproducibility & Device Setup**

In [2]:
SEED = 42

def set_seed(seed: int = SEED) -> None:
    """Fix all relevant random seeds for reproducible results."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

Using device: cpu


## **Load Processed Data**

In [3]:
DATA_PATH = "../data/processed/processed_data.pkl"

with open(DATA_PATH, "rb") as f:
    processed = pickle.load(f)

features   = processed["features"]      # shape: [num_nodes, 128]
labels     = processed["labels"]        # shape: [num_nodes]
edge_index = processed["edge_index"]    # shape: [2, num_edges]
train_idx  = processed["train_idx"]
valid_idx  = processed["valid_idx"]
test_idx   = processed["test_idx"]

num_classes = int(labels.max().item()) + 1
num_features = features.shape[1]

# Wrap everything in a single PyG Data object — the standard
# container GNN layers expect (x, edge_index, y together).
graph_data = Data(
    x=features,
    edge_index=edge_index,
    y=labels,
).to(DEVICE)

print(f"Nodes: {graph_data.num_nodes:,}")
print(f"Edges: {graph_data.num_edges:,}")
print(f"Features per node: {num_features}")
print(f"Number of classes: {num_classes}")
print(f"Train / Valid / Test sizes: {len(train_idx):,} / {len(valid_idx):,} / {len(test_idx):,}")

Nodes: 169,343
Edges: 1,166,243
Features per node: 128
Number of classes: 40
Train / Valid / Test sizes: 90,941 / 29,799 / 48,603


## **Model 1: Graph Convolutional Network (GCN)**

In [4]:
class GCN(nn.Module):
    """
    Baseline spectral-style GNN (Kipf & Welling, 2017).

    Architecture:
        Input -> GCNConv -> BatchNorm -> ReLU -> Dropout
              -> GCNConv -> BatchNorm -> ReLU -> Dropout
              -> GCNConv -> logits

    Design choices:
        - 3 layers: enough to aggregate 3-hop neighborhood info
          without over-smoothing (a known GCN failure mode at
          higher depth on citation graphs).
        - BatchNorm after each conv, before activation: stabilizes
          training on this dataset specifically — matches the
          standard OGB baseline implementation, which relies on
          BatchNorm to reach its reported ~71.7% test accuracy.
          Without it, feature scale drift across layers slows
          convergence and caps achievable accuracy.
        - ReLU: standard, cheap, avoids vanishing gradients
          better than sigmoid/tanh for this depth.
        - Dropout after each hidden layer: GCNs are prone to
          overfitting on small-label-fraction node classification
          tasks; dropout regularizes without hurting message passing.
    """

    def __init__(
        self,
        in_channels: int,
        hidden_channels: int,
        out_channels: int,
        dropout: float = 0.5,
    ):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.bn1 = nn.BatchNorm1d(hidden_channels)

        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.bn2 = nn.BatchNorm1d(hidden_channels)

        self.conv3 = GCNConv(hidden_channels, out_channels)
        self.dropout = dropout

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.conv3(x, edge_index)  # raw logits, softmax applied in loss
        return x

### **Model 2: GraphSAGE**

In [5]:
class GraphSAGE(nn.Module):
    """
    Inductive, neighbor-sampling-friendly GNN (Hamilton et al., 2017).

    Architecture:
        Input -> SAGEConv -> BatchNorm -> ReLU -> Dropout
              -> SAGEConv -> BatchNorm -> ReLU -> Dropout
              -> SAGEConv -> logits

    Design choices vs. GCN:
        - SAGEConv aggregates neighbor features via a learned
          aggregation (mean, by default here) rather than a fixed
          spectral normalization — makes it inductive (can
          generalize to unseen nodes/graphs, unlike vanilla GCN).
        - Chosen as the second architecture specifically because
          it scales to large graphs (169K nodes here) via
          mini-batch neighbor sampling, which plain GCN does not
          support natively.
        - BatchNorm added for the same stabilization reason as GCN,
          keeping the comparison in Task 06 fair (both models get
          identical regularization treatment, isolating the effect
          of the aggregation mechanism, not capacity).
    """

    def __init__(
        self,
        in_channels: int,
        hidden_channels: int,
        out_channels: int,
        dropout: float = 0.5,
    ):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.bn1 = nn.BatchNorm1d(hidden_channels)

        self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        self.bn2 = nn.BatchNorm1d(hidden_channels)

        self.conv3 = SAGEConv(hidden_channels, out_channels)
        self.dropout = dropout

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.conv3(x, edge_index)
        return x

### **Hyperparameter Configuration & Model Instantiation**

In [6]:
CONFIG = {
    "hidden_channels": 256,
    "dropout": 0.5,
    "in_channels": num_features,
    "out_channels": num_classes,
}

gcn_model = GCN(
    in_channels=CONFIG["in_channels"],
    hidden_channels=CONFIG["hidden_channels"],
    out_channels=CONFIG["out_channels"],
    dropout=CONFIG["dropout"],
).to(DEVICE)

sage_model = GraphSAGE(
    in_channels=CONFIG["in_channels"],
    hidden_channels=CONFIG["hidden_channels"],
    out_channels=CONFIG["out_channels"],
    dropout=CONFIG["dropout"],
).to(DEVICE)

def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("GCN trainable parameters:      ", f"{count_parameters(gcn_model):,}")
print("GraphSAGE trainable parameters:", f"{count_parameters(sage_model):,}")

GCN trainable parameters:       110,120
GraphSAGE trainable parameters: 218,664


### **Sanity Check: Forward Pass Verification**

In [7]:
gcn_model.eval()
sage_model.eval()

with torch.no_grad():
    start = time.time()
    gcn_out = gcn_model(graph_data.x, graph_data.edge_index)
    gcn_time = time.time() - start

    start = time.time()
    sage_out = sage_model(graph_data.x, graph_data.edge_index)
    sage_time = time.time() - start

assert gcn_out.shape == (graph_data.num_nodes, num_classes), "GCN output shape mismatch"
assert sage_out.shape == (graph_data.num_nodes, num_classes), "GraphSAGE output shape mismatch"

print(f"GCN output shape:       {tuple(gcn_out.shape)}  |  forward pass: {gcn_time:.2f}s")
print(f"GraphSAGE output shape: {tuple(sage_out.shape)}  |  forward pass: {sage_time:.2f}s")
print("\nBoth models produce correctly-shaped logits — ready for training in Task 05.")

GCN output shape:       (169343, 40)  |  forward pass: 2.58s
GraphSAGE output shape: (169343, 40)  |  forward pass: 1.87s

Both models produce correctly-shaped logits — ready for training in Task 05.


### **Save Model Configs for Task 05 Handoff**

In [8]:
# Save Architecture Config

import json
from pathlib import Path

def find_project_root(marker: str = "requirements.txt") -> Path:
    """Walk up from the current working directory until we find
    the project root, identified by the presence of a known file."""
    current = Path.cwd()
    for parent in [current] + list(current.parents):
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"Could not locate project root (missing {marker})")

PROJECT_ROOT = find_project_root()
CONFIG_PATH = PROJECT_ROOT / "models_checkpoints" / "model_config.json"

CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)

model_config = {
    "gcn": {"class": "GCN", **CONFIG},
    "graphsage": {"class": "GraphSAGE", **CONFIG},
    "seed": SEED,
}

with open(CONFIG_PATH, "w") as f:
    json.dump(model_config, f, indent=2)

print(f"Saved model_config.json to: {CONFIG_PATH}")
print(json.dumps(model_config, indent=2))

Saved model_config.json to: C:\Users\thrit\Desktop\ogbn-arxiv-gnn\models_checkpoints\model_config.json
{
  "gcn": {
    "class": "GCN",
    "hidden_channels": 256,
    "dropout": 0.5,
    "in_channels": 128,
    "out_channels": 40
  },
  "graphsage": {
    "class": "GraphSAGE",
    "hidden_channels": 256,
    "dropout": 0.5,
    "in_channels": 128,
    "out_channels": 40
  },
  "seed": 42
}
